# Input/Output con `pandas`

Para poder leer o escribir datos, `pandas` provee una serie de métodos específicos para todos los tipos de datos estructurados usuales, ya sea en formato texto o binario. Los métodos para leer de archivos se reconocen por el prefijo `.read_`, mientras que para escribir usaremos `.to_`. Para cada formato particular de archivo, estos métodos aceptarán un conjunto de argumentos adicionales que permiten adecuar nuestro código. 

> Si se usa VSCode, se puede instalar la extensión [Data Wrangler](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.datawrangler) que permite inspeccionar `DataFrames`. Puede ser útil cuando uno trabaja con muchos datos.

## JSON

JSON (JavaScript Object Notation) es un formato estándar de estructura de datos en modo texto, legible y de amplio uso en internet. 

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
atomos_path = Path.cwd() / 'data' / 'atomos'
print(atomos_path)

/Users/flavioc/Codes/GitHub/cursos/diplo-ia/etl-02/data/atomos


In [3]:
df = pd.read_json(atomos_path / 'atomos.json')
df

,Element,Symbol,Atomic Number,Atomic Mass (u),Density (g/cm³),Melting Point (K),Boiling Point (K),Electronegativity,State at Room Temp
0,Hydrogen,H,1,1.0080,0.000090,14.01,20.28,2.20,Gas
1,Helium,He,2,4.0026,0.000179,NaN,NaN,NaN,Gas
2,Lithium,Li,3,6.9400,0.534000,453.69,1615.00,0.98,Solid
3,Beryllium,Be,4,9.0122,1.850000,1560.00,2742.00,1.57,Solid
4,Boron,B,5,10.8100,2.340000,2349.00,4200.00,2.04,Solid
5,Carbon,C,6,12.0110,2.267000,3800.00,4300.00,2.55,Solid
6,Nitrogen,N,7,14.0070,0.001251,63.15,77.36,3.04,Gas
7,Oxygen,O,8,15.9990,0.001429,54.36,90.20,3.44,Gas
8,Fluorine,F,9,18.9980,0.001696,53.48,85.03,3.98,Gas
9,Neon,Ne,10,20.1800,0.000900,NaN,NaN,NaN,Gas


In [5]:
df.dtypes

Element                   str
Symbol                    str
Atomic Number           int64
Atomic Mass (u)       float64
Density (g/cm³)       float64
Melting Point (K)     float64
Boiling Point (K)     float64
Electronegativity     float64
State at Room Temp        str
dtype: object

In [6]:
df.columns

Index(['Element', 'Symbol', 'Atomic Number', 'Atomic Mass (u)',
       'Density (g/cm³)', 'Melting Point (K)', 'Boiling Point (K)',
       'Electronegativity', 'State at Room Temp'],
      dtype='str')

Renombremos algunas columnas para que no tengan etiquetas tan complejas

In [6]:
df.rename(columns={'Density (g/cm³)': 'Density'}, inplace=True)
df.rename(columns={'Melting Point (K)': 'Melting Point'}, inplace=True)
df.columns

Index(['Element', 'Symbol', 'Atomic Number', 'Atomic Mass (u)', 'Density',
       'Melting Point', 'Boiling Point (K)', 'Electronegativity',
       'State at Room Temp'],
      dtype='str')

Extraigamos los datos atómicos (masa y número atómico) y escribámoslo en un json:

In [7]:
df[['Element', 'Symbol', 'Atomic Number', 'Atomic Mass (u)']]

,Element,Symbol,Atomic Number,Atomic Mass (u)
0,Hydrogen,H,1,1.0080
1,Helium,He,2,4.0026
2,Lithium,Li,3,6.9400
3,Beryllium,Be,4,9.0122
4,Boron,B,5,10.8100
5,Carbon,C,6,12.0110
6,Nitrogen,N,7,14.0070
7,Oxygen,O,8,15.9990
8,Fluorine,F,9,18.9980
9,Neon,Ne,10,20.1800


In [8]:
df[['Element', 'Symbol', 'Atomic Number', 'Atomic Mass (u)']].to_json(atomos_path / 'prop_atomos.json')

In [9]:
# Guarda sólo los valores
df[['Element', 'Symbol', 'Atomic Number', 'Atomic Mass (u)']].to_json(atomos_path / 'prop_atomos_values.json', orient='values')

# Guarda los registros indexados por el índice del DataFrame
df[['Element', 'Symbol', 'Atomic Number', 'Atomic Mass (u)']].to_json(atomos_path / 'prop_atomos_index.json', orient='index')

# Guarda los registros indexados por el nombre de las columnas, valor por defecto
df[['Element', 'Symbol', 'Atomic Number', 'Atomic Mass (u)']].to_json(atomos_path / 'prop_atomos_columns.json', orient='columns')

# Guarda los registros en formato de lista de diccionarios, sin indices
df[['Element', 'Symbol', 'Atomic Number', 'Atomic Mass (u)']].to_json(atomos_path / 'prop_atomos_records.json', orient='records')


## CSV

Podríamos querer escribirlos como valores separados por comas (u otro delimitador):

In [10]:
df[['Element', 'Symbol', 'Atomic Number', 'Atomic Mass (u)']].to_csv(atomos_path / 'prop_atomos.csv', sep='|')

Vemos que en ambos casos también guarda el índice en el archivo

Si queremos que no guarde el índice, pasamos el argumento opcional `index=False`. 

In [11]:
df.to_csv(atomos_path / 'prop_atomos_noindex.csv', sep='|', index=False)

Se puede ver que al guardar el `.csv`, los elementos inexistentes de la tabla (`NaN`) se guardan como cadenas de caracteres vacías. Se puede indicar otro tipo de valor para esos casos:

In [12]:
df.to_csv(atomos_path / 'prop_atomos_noheader2.csv', sep='|', index=False, na_rep='N/A')

## Formatos binarios

En algunos casos como leer o escribir de archivos binarios, es necesario instalar algunos módulos. Por ejemplo, si queremos leer archivos `.parquet`, tenemos que instalar `pyarrow`.

- `.parquet`: `conda install pyarrow`
- `.xls(x)`: `conda install openpyxl xlrd`
- `hdf5`: `conda install pytables`

### El formato `parquet`

In [13]:
df.to_parquet(atomos_path / 'prop_atomos.parquet')

In [14]:
dfp = pd.read_parquet(atomos_path / 'prop_atomos.parquet')
dfp

,Element,Symbol,Atomic Number,Atomic Mass (u),Density (g/cm³),Melting Point (K),Boiling Point (K),Electronegativity,State at Room Temp
0,Hydrogen,H,1,1.0080,0.000090,14.01,20.28,2.20,Gas
1,Helium,He,2,4.0026,0.000179,NaN,NaN,NaN,Gas
2,Lithium,Li,3,6.9400,0.534000,453.69,1615.00,0.98,Solid
3,Beryllium,Be,4,9.0122,1.850000,1560.00,2742.00,1.57,Solid
4,Boron,B,5,10.8100,2.340000,2349.00,4200.00,2.04,Solid
5,Carbon,C,6,12.0110,2.267000,3800.00,4300.00,2.55,Solid
6,Nitrogen,N,7,14.0070,0.001251,63.15,77.36,3.04,Gas
7,Oxygen,O,8,15.9990,0.001429,54.36,90.20,3.44,Gas
8,Fluorine,F,9,18.9980,0.001696,53.48,85.03,3.98,Gas
9,Neon,Ne,10,20.1800,0.000900,NaN,NaN,NaN,Gas


### El formato `xls`

 `.xls` es un formato binario, el formato antiguo de Excel.

En cambio:

- `.xls`: binario
- `.xlsx`: archivo ZIP con XML adentro
- `.xlsm`: como `.xlsx`, pero con macros
- `.csv`: texto plano

En pandas, para leer `.xls` normalmente se necesita un engine compatible como `xlrd`, mientras que para `.xlsx` se suele usar `openpyxl`.

In [15]:
df.to_excel(atomos_path / 'prop_atomos.xlsx', index=False)

In [16]:
dfx = pd.read_excel(atomos_path / 'prop_atomos.xlsx')
dfx

,Element,Symbol,Atomic Number,Atomic Mass (u),Density (g/cm³),Melting Point (K),Boiling Point (K),Electronegativity,State at Room Temp
0,Hydrogen,H,1,1.0080,0.000090,14.01,20.28,2.20,Gas
1,Helium,He,2,4.0026,0.000179,NaN,NaN,NaN,Gas
2,Lithium,Li,3,6.9400,0.534000,453.69,1615.00,0.98,Solid
3,Beryllium,Be,4,9.0122,1.850000,1560.00,2742.00,1.57,Solid
4,Boron,B,5,10.8100,2.340000,2349.00,4200.00,2.04,Solid
5,Carbon,C,6,12.0110,2.267000,3800.00,4300.00,2.55,Solid
6,Nitrogen,N,7,14.0070,0.001251,63.15,77.36,3.04,Gas
7,Oxygen,O,8,15.9990,0.001429,54.36,90.20,3.44,Gas
8,Fluorine,F,9,18.9980,0.001696,53.48,85.03,3.98,Gas
9,Neon,Ne,10,20.1800,0.000900,NaN,NaN,NaN,Gas


## Leyendo HTML

Eventualmente, es posible que uno tenga que leer archivos HTML, por ejemplo, si uno tiene que bajar datos de la web.
Probablemente se necesite el módulo `lxml` (`conda instal lxml`).

In [17]:
url = "https://en.wikipedia.org/wiki/List_of_countries_by_population"

tablas = pd.read_html(url)
len(tablas)

HTTPError: HTTP Error 403: Forbidden


> En el ejemplo, tenemos que usar el módulo `requests` para traer el html, porque si usamos la función de pandas `read_html` directamente, la página nos responde `403 Forbidden`...

Tenemos que instalar `requests`:
```bash
conda install requests
```


In [18]:
import requests
from io import StringIO

# A veces, los sitios web bloquean las solicitudes que no parecen provenir de un navegador real. Para evitar esto, podemos agregar un encabezado User-Agent a nuestra solicitud.
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.raise_for_status()


In [19]:
response.text[:500]  # Mostrar los primeros 500 caracteres del contenido de la página

'<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientp'

In [20]:
# pd.read_html no siempre interpreta un string HTML como HTML.
# StringIO lo convierte en un objeto tipo archivo y evita que pandas lo trate como una ruta.
tablas = pd.read_html(StringIO(response.text)) #, flavor="bs4")
len(tablas)


3

In [21]:
# Inspeccionamos las tablas encontradas antes de elegir una.
for i, tabla in enumerate(tablas):
    print(i, tabla.shape, list(tabla.columns)[:3])


0 (240, 6) ['Location', 'Population', '% of world']
1 (13, 2) ['vteLists of countries by population statistics', 'vteLists of countries by population statistics.1']
2 (1, 2) ['vteWorld', 'vteWorld.1']


In [22]:
print(f"Encontré {len(tablas)} tablas")
print(type(tablas))

# La función devuelve una lista de DataFrames, cada uno correspondiente a una tabla en la página.
tabla_poblacion = tablas[0]

tabla_poblacion.head()


Encontré 3 tablas
<class 'list'>


,Location,Population,% of world,Date,Source (official or from the United Nations),Notes
0,World,8232000000,100%,13 Jun 2025,UN projection[1][3],NaN
1,India,1417492000,17.2%,1 Jul 2025,Official projection[4],[b]
2,China,1404890000,17.0%,31 Dec 2025,Official estimate[5],[c]
3,United States,341784857,4.1%,1 Jul 2025,Official estimate[6],[d]
4,Indonesia,288315089,3.5%,31 Dec 2025,National annual projection[7],NaN


## Leyendo datos desde una API pública

Cuando una institución publica una API, suele ser preferible consumir esa fuente antes que extraer tablas desde HTML. La respuesta normalmente viene en JSON, que podemos convertir a un `DataFrame`.

En este ejemplo usamos la [API de estadísticas cambiarias del BCRA](https://www.bcra.gob.ar/documentacion-apis/?fileName=estadisticascambiarias-v1). Si no enviamos una fecha, la API devuelve la última cotización disponible.

In [23]:
url_bcra = "https://api.bcra.gob.ar/estadisticascambiarias/v1.0/Cotizaciones"
params = {"fecha": "2026-06-16"}

response = requests.get(url_bcra, params=params)
response.raise_for_status()

datos = response.json()
datos.keys()

dict_keys(['status', 'results'])

In [ ]:
datos

In [24]:
datos["results"].keys()

dict_keys(['fecha', 'detalle'])

In [26]:
cotizaciones = pd.json_normalize(datos["results"]["detalle"])
cotizaciones["fecha"] = pd.to_datetime(datos["results"]["fecha"])

cotizaciones.head()

,codigoMoneda,descripcion,tipoPase,tipoCotizacion,fecha
0,ARS,PESO,0.000696,0.000000,2026-06-16
1,AUD,DOLAR AUSTRALIA,0.707300,1016.036450,2026-06-16
2,AWG,FLORIN (ANTILLAS HOLANDESAS),0.558659,802.513966,2026-06-16
3,BOB,BOLIVIANOS,0.145117,208.460311,2026-06-16
4,BRL,REAL,0.196641,282.475322,2026-06-16


In [27]:
monedas = ["USD", "EUR", "BRL", "CLP"]
columnas = ["fecha", "codigoMoneda", "descripcion", "tipoCotizacion"]

cotizaciones.loc[cotizaciones["codigoMoneda"].isin(monedas), columnas]

,fecha,codigoMoneda,descripcion,tipoCotizacion
4,2026-06-16,BRL,REAL,282.475322
7,2026-06-16,CLP,PESO CHILENO,1.620673
13,2026-06-16,EUR,EURO,1668.207450
31,2026-06-16,USD,DOLAR E.E.U.U.,1436.500000
